# Hogbom Deconvolve Demonstrator

This notebook demonstrates the implementation of Hogbom clean within the astroviper framework. 
CASA is used to generate the initial set of images (residual, PSF). The CASA task `deconvolve` is run to deconvolve the residual image. astroviper Hogbom is also run on a copy of the same images, and the two outputs are compared. 

This notebook is organized into three sections : The first section runs the CASA deconvolution, the second the astroviper deconvolution, and finally the comparison images and plots. 

This has to be done, because importing CASA alongside xradio causes some namespace conflicts and segmentation faults, so once we are done with all the CASA functionality, we will import the astroviper libraries.

In [ ]:
# Common imports across all sections

import glob
import os
import shutil
import ssl
import tarfile
import urllib

import certifi
import numpy as np
import toolviper

## Download images via toolviper

Wipe any images with the same name on disk, and pull the images from the toolviper data repo in order to make the comparison between astroviper hogbom clean and CASA hogbom clean

In [ ]:
def wipe_images(imlist):
    for im in imlist:
        if os.path.exists(im):
            shutil.rmtree(im)

In [ ]:
resid_orig = "test_hogbom_multisrc.residual"
psf_orig = "test_hogbom_multisrc.psf"
casa_deconv_resid = "casa_hogbom_deconv.residual"

wipe_images([resid_orig, psf_orig, casa_deconv_resid])

# Update data manifest always
toolviper.utils.data.update()
toolviper.utils.data.download([resid_orig, psf_orig, casa_deconv_resid])

## NOTE

The progress bars on the toolviper download may not appear complete, which is a bug in the Jupyter display. The files are very small (< 1 MB total) and the download completes relatively rapidly.

## Astroviper : Load images, run deconvolution

Import the astroviper and xradio libraries and load the copies of the residual and PSF images generated by CASA.

The current `deconvolve` API works on a single image dataset and updates it *in place*: it reads the residual and PSF referenced by the input data group, subtracts the CLEAN components from the residual, and accumulates them into a `SKY_MODEL` variable. We therefore combine the residual and PSF into one dataset (`img_xds`), register a `"residual"` data group, and call `deconvolve(img_xds, algorithm="hogbom", deconvolve_params=...)`.

In [ ]:
# Import astroviper libraries
try:
    from xradio.image import load_image, open_image

    from astroviper.processing_functions.image_analysis.point_spread_function_gaussian_fit import (
        point_spread_function_gaussian_fit,
    )
    from astroviper.processing_functions.imaging.deconvolution import deconvolve
except ImportError:
    print(
        "Please install the astroviper libraries. This can be done via pip install astroviper, "
        "or cloning the git repository (https://github.com/casangi/astroviper/) and installing "
        "the relevant branch."
    )

# Load the CASA-generated residual and PSF images, plus the reference
# CASA-deconvolved residual used for comparison below.
#
# ``.load()`` pulls the arrays into memory as plain numpy. The current
# deconvolver updates the residual and model *in place*, which requires
# in-memory (non-dask) buffers -- without ``.load()`` the CLEAN components
# would be written into throwaway copies.
resid_xds = load_image(resid_orig).load()
psf_xds = load_image(psf_orig).load()
casa_residual_xds = load_image(casa_deconv_resid).load()

In [ ]:
# The current deconvolve() API operates on a *single* image dataset, in place.
# It reads the residual and PSF referenced by the input data group, subtracts
# CLEAN components from the residual, and accumulates them into a model image.
#
# Build that combined dataset from the residual and PSF images, and register a
# "residual" data group describing which variables hold the sky residual and
# the PSF. (resid_xds is left untouched -- .copy() is deep -- so it still holds
# the original dirty image for the comparison plots below.)
img_xds = resid_xds.copy()
img_xds["POINT_SPREAD_FUNCTION"] = psf_xds["POINT_SPREAD_FUNCTION"]
img_xds.attrs["data_groups"] = {
    "residual": {
        "sky": "RESIDUAL",
        "point_spread_function": "POINT_SPREAD_FUNCTION",
    }
}

# Fit the PSF main lobe and record its maximum sidelobe level. This adds a
# MAX_SIDELOBE_POINT_SPREAD_FUNCTION variable to the "residual" data group,
# which the deconvolver reports in its per-plane return statistics.
img_xds = point_spread_function_gaussian_fit(
    img_xds,
    image_data_group_in_name="residual",
    image_data_group_out_name="residual",
    overwrite=True,
)

# Run Hogbom CLEAN. deconvolve() mutates img_xds in place: RESIDUAL has the
# CLEAN components subtracted from it, and a new SKY_MODEL variable accumulates
# them. It returns a per-plane statistics ReturnDict (not a 3-tuple).
returndict = deconvolve(
    img_xds=img_xds,
    algorithm="hogbom",
    deconvolve_params={"gain": 0.1, "niter": 501, "threshold": 0},
)

In [ ]:
# Per-plane CLEAN statistics: iterations performed, peak residual before/after
# (start_peakres -> peakres), the cleaned model flux, PSF sidelobe, etc.
print(returndict)

In [ ]:
# Peak of the astroviper residual after CLEAN (updated in place on img_xds).
print(np.squeeze(img_xds["RESIDUAL"].values).max())

## Results

Compare the results of the astroviper and CASA outputs. Residual images after deconvolution, model images, fractional differences.

In [ ]:
try:
    import holoviews as hv
except ImportError:
    print(f"Please install holoviews. pip install holoviews")
    exit

hv.extension("bokeh")

In [ ]:
img0 = hv.Image(np.squeeze(resid_xds["RESIDUAL"].values)).opts(
    tools=["hover"], title="Original Dirty Image", width=500, height=500, colorbar=True
)

In [ ]:
img1 = hv.Image(np.squeeze(img_xds["RESIDUAL"].values)).opts(
    tools=["hover"],
    title="Astroviper residual image",
    width=300,
    height=300,
    colorbar=True,
)
img2 = hv.Image(np.squeeze(np.squeeze(casa_residual_xds["RESIDUAL"].values))).opts(
    tools=["hover"], title="CASA residual image", width=300, height=300, colorbar=True
)

diff = np.squeeze(casa_residual_xds["RESIDUAL"].values) - np.squeeze(
    img_xds["RESIDUAL"].values
)
img3 = hv.Image(diff).opts(
    tools=["hover"],
    title="Difference residual image",
    width=500,
    height=500,
    colorbar=True,
)

# Astroviper CLEAN model (SKY_MODEL), created in place on img_xds.
img_model = hv.Image(np.squeeze(img_xds["SKY_MODEL"].values)).opts(
    tools=["hover"],
    title="Astroviper model image",
    width=300,
    height=300,
    colorbar=True,
)

In [ ]:
img0

In [ ]:
grid = hv.Layout([img1, img2, img_model])
grid

In [ ]:
img3